# Process ACS data

Outcomes:
- Load and process ACS data obtained from data.census.gov
- This example provides two preset functions for processing income and age groups

In [1]:
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2
from helpers import acs

In [ ]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/acs"
OUTPUT_DIR = "data/processed"

# ACS table files
CBSA_CODE = 38060
INCOME_FILE = f"{DATA_DIR}/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B19001-Data.csv"   # household income
AGE_FILE = f"{DATA_DIR}/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B01001-Data.csv"      # sex by age

# outputs
OUT_PATH_ACS_DISTR = f"{OUTPUT_DIR}/acs_cbg_distr_{CBSA_CODE}_test.csv"
OUT_PATH_INCOME_MARGINS = f"{OUTPUT_DIR}/acs_income_margins_{CBSA_CODE}_test.csv"
OUT_PATH_AGE_MARGINS = f"{OUTPUT_DIR}/acs_age_margins_{CBSA_CODE}_test.csv"

# ======================
# INCOME GROUPING (B19001)
# ======================

INCOME_GROUPS = {
    "<35k": [
        "B19001_002E", 
        "B19001_003E", 
        "B19001_004E",
        "B19001_005E", 
        "B19001_006E", 
        "B19001_007E",
    ],

    "35k-75k": [
        "B19001_008E", 
        "B19001_009E",
        "B19001_010E", 
        "B19001_011E", 
        "B19001_012E",
    ],

    "75k-100k": [
        "B19001_013E",
    ],

    "100k+": [
        "B19001_014E", 
        "B19001_015E",
        "B19001_016E", 
        "B19001_017E",
    ],
}

# ======================
# AGE GROUPING (B01001)
# ======================

AGE_GROUPS = {

    "Under 18": [
        "Under 5 years", 
        "5 to 9 years",
        "10 to 14 years", 
        "15 to 17 years",
    ],

    "18 to 24": [
        "18 and 19 years", 
        "20 years",
        "21 years", 
        "22 to 24 years",
    ],

    "25 to 44": [
        "25 to 29 years", 
        "30 to 34 years",
        "35 to 39 years", 
        "40 to 44 years",
    ],

    "45 to 66": [
        "45 to 49 years", 
        "50 to 54 years",
        "55 to 59 years", 
        "60 and 61 years",
        "62 to 64 years", 
        "65 and 66 years",
    ],

    "67+": [
        "67 to 69 years", 
        "70 to 74 years",
        "75 to 79 years", 
        "80 to 84 years",
        "85 years and over",
    ],
}

# drop groups (e.g. if mobility sample only contains 18+ users)
DROP_AGE_GROUPS = ["Under 18"]

# ======================
# NOTES
# ======================
# - Paths assume CSV downloads from data.census.gov
# - skiprows behavior handled in processing functions
# - order of group labels must align with downstream calibration targets

In [7]:
income_cbg, income_cbsa = acs.process_income_table(INCOME_FILE, INCOME_GROUPS, last_row_margins=True)
income_cbsa = acs.format_cbsa_marginals(income_cbsa, var_name="hh_income")

In [8]:
income_cbg

,GEOID,<35k,35k-75k,75k-100k,100k+
0,040130101021,0.085106,0.12766,0.182033,0.605201
1,040130101022,0.295756,0.087533,0.049072,0.567639
2,040130101023,0.149068,0.10352,0.064182,0.68323
3,040130101031,0.22651,0.275168,0.082215,0.416107
4,040130101032,0.118758,0.140351,0.17004,0.57085
...,...,...,...,...,...
3028,040219413001,0.680303,0.157576,0.107576,0.054545
3029,040219414011,0.319481,0.27013,0.044156,0.366234
3030,040219414012,0.585915,0.273239,0.056338,0.084507
3031,040219414013,0.243065,0.459709,0.228534,0.068692


In [ ]:
income_cbsa # target marginals for the CBSA

,hh_income,pop
0,<35k,416253.0
1,35k-75k,547471.0
2,75k-100k,237884.0
3,100k+,543611.0


In [10]:
age_cbg, age_cbsa = acs.process_age_table(AGE_FILE, AGE_GROUPS, DROP_AGE_GROUPS, last_row_margins=True)
age_cbsa = acs.format_cbsa_marginals(age_cbsa, var_name="age_group")

In [12]:
age_cbg

,GEOID,18 to 24,25 to 44,45 to 66,67+
0,040130101021,0.003542,0.319953,0.279811,0.396694
1,040130101022,0.006592,0.071852,0.555043,0.366513
2,040130101023,0.023019,0.0,0.464151,0.51283
3,040130101031,0.045374,0.129004,0.55694,0.268683
4,040130101032,0.026382,0.236809,0.499372,0.237437
...,...,...,...,...,...
3028,040219413001,0.25635,0.38925,0.264028,0.090372
3029,040219414011,0.186841,0.299849,0.331994,0.181316
3030,040219414012,0.199516,0.408706,0.274486,0.117291
3031,040219414013,0.192424,0.283333,0.432323,0.091919


In [11]:
age_cbsa

,age_group,pop
0,18 to 24,444363.0
1,25 to 44,1337601.0
2,45 to 66,1262610.0
3,67+,663574.0


In [15]:
cbg_distr = acs.merge_distr_tables([age_cbg, income_cbg], on='GEOID', how='inner', dropna=True)
cbg_distr

,GEOID,18 to 24,25 to 44,45 to 66,67+,<35k,35k-75k,75k-100k,100k+
0,040130101021,0.003542,0.319953,0.279811,0.396694,0.085106,0.12766,0.182033,0.605201
1,040130101022,0.006592,0.071852,0.555043,0.366513,0.295756,0.087533,0.049072,0.567639
2,040130101023,0.023019,0.0,0.464151,0.51283,0.149068,0.10352,0.064182,0.68323
3,040130101031,0.045374,0.129004,0.55694,0.268683,0.22651,0.275168,0.082215,0.416107
4,040130101032,0.026382,0.236809,0.499372,0.237437,0.118758,0.140351,0.17004,0.57085
...,...,...,...,...,...,...,...,...,...
3028,040219413001,0.25635,0.38925,0.264028,0.090372,0.680303,0.157576,0.107576,0.054545
3029,040219414011,0.186841,0.299849,0.331994,0.181316,0.319481,0.27013,0.044156,0.366234
3030,040219414012,0.199516,0.408706,0.274486,0.117291,0.585915,0.273239,0.056338,0.084507
3031,040219414013,0.192424,0.283333,0.432323,0.091919,0.243065,0.459709,0.228534,0.068692


In [ ]:
cbg_distr.to_csv(OUT_PATH_ACS_DISTR, index=False)
income_cbsa.to_csv(OUT_PATH_INCOME_MARGINS, index=False)
age_cbsa.to_csv(OUT_PATH_AGE_MARGINS, index=False)